# Synthetic Student Residence Data Generator

Generates realistic synthetic student residence records using the **Anthropic Claude API**.

```

In [1]:
!pip install anthropic gradio python-dotenv --quiet

In [2]:
import os
import json
import anthropic
import gradio as gr
from dotenv import load_dotenv

load_dotenv(override=True)

MODEL      = "claude-sonnet-4-20250514"
MAX_TOKENS = 4096

print("Imports loaded")

Imports loaded


In [3]:
def get_client(api_key: str = ""):
    key = api_key.strip() or os.environ.get("ANTHROPIC_API_KEY", "")
    if not key:
        raise ValueError("No API key found. Set ANTHROPIC_API_KEY or enter it in the UI.")
    return anthropic.Anthropic(api_key=key)

print("Client helper ready")

Client helper ready


In [4]:
SYSTEM_PROMPT = """
You are a synthetic data generator for student residence management systems.
When given a prompt, generate realistic but entirely fictional student residence
records. Always respond with valid, well-formatted JSON only — no extra text,
no markdown fences, no explanation. All names, IDs, and personal details must
be completely fictional.
""".strip()

print("System prompt set")

System prompt set


In [5]:
def generate_synthetic_data(prompt: str, api_key: str = "") -> str:
    """
    Send a prompt to Claude and return generated synthetic data as a JSON string.
    """
    client = get_client(api_key)

    message = client.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": prompt}]
    )

    raw = message.content[0].text.strip()

    # Strip markdown fences if model adds them
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[-1]
        raw = raw.rsplit("```", 1)[0].strip()

    # Pretty-print if valid JSON
    try:
        return json.dumps(json.loads(raw), indent=2)
    except json.JSONDecodeError:
        return raw

print("Generation function ready")

Generation function ready


In [6]:
# ── Preset prompts ────────────────────────────────────────────────────────

PROMPT_STUDENTS = """
Generate 5 synthetic student residence records in JSON format.
Each record must include:
- student_id
- full_name
- age
- gender
- nationality
- email
- phone_number
- residence_hall
- room_number
- room_type  (single / double / suite)
- floor
- check_in_date
- check_out_date
- lease_status  (active / expired / pending)
- monthly_fee_usd
- meal_plan  (full / partial / none)
- emergency_contact_name
- emergency_contact_phone
""".strip()

PROMPT_MAINTENANCE = """
Generate 5 synthetic student residence maintenance request records in JSON format.
Each record must include:
- request_id
- student_id
- student_name
- room_number
- residence_hall
- issue_category  (plumbing / electrical / HVAC / furniture / internet / pest / other)
- issue_description
- priority  (low / medium / high / urgent)
- date_submitted
- date_resolved
- status  (open / in_progress / resolved / cancelled)
- assigned_technician
- resolution_notes
""".strip()

PROMPT_INCIDENTS = """
Generate 5 synthetic student residence incident reports in JSON format.
Each record must include:
- incident_id
- student_id
- student_name
- residence_hall
- room_number
- incident_type  (noise / property_damage / visitor_policy / fire_safety / substance / other)
- description
- date_reported
- reported_by
- outcome  (warning / fine / probation / eviction / no_action)
- fine_amount_usd
- appeal_status  (none / pending / upheld / overturned)
""".strip()

PROMPT_BILLING = """
Generate 5 synthetic student residence billing records in JSON format.
Each record must include:
- invoice_id
- student_id
- student_name
- residence_hall
- room_number
- billing_period
- room_charge_usd
- meal_plan_charge_usd
- laundry_charge_usd
- damage_deposit_usd
- outstanding_fines_usd
- total_due_usd
- due_date
- payment_status  (paid / partial / overdue / pending)
- payment_method  (bank_transfer / credit_card / bursary / cash)
""".strip()

print("Preset prompts ready")

Preset prompts ready


In [7]:

API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")

if API_KEY:
    print("Generating sample student residence records...\n")
    result = generate_synthetic_data(PROMPT_STUDENTS, api_key=API_KEY)
    print(result)
else:
    print("No ANTHROPIC_API_KEY found. Use the Gradio UI cell below.")

No ANTHROPIC_API_KEY found. Use the Gradio UI cell below.


In [10]:


PRESET_MAP = {
    "Student Records":      PROMPT_STUDENTS,
    "Maintenance Requests": PROMPT_MAINTENANCE,
    "Incident Reports":     PROMPT_INCIDENTS,
    "Billing Records":      PROMPT_BILLING,
    "Custom":               "",
}

def generate_ui(api_key, data_type, custom_prompt, num_records):
    base = PRESET_MAP.get(data_type, PROMPT_STUDENTS)
    if data_type == "Custom":
        base = custom_prompt.strip()
    if not base:
        return "Please enter a custom prompt."
    prompt = base.replace("Generate 5", f"Generate {int(num_records)}")
    try:
        return generate_synthetic_data(prompt, api_key=api_key)
    except Exception as e:
        return f"Error: {e}"


with gr.Blocks(title="Synthetic Student Residence Data Generator") as demo:

    gr.Markdown("# Synthetic Student Residence Data Generator")
    gr.Markdown("Generate realistic fictional student residence data for testing and development. All data is entirely synthetic.")

    api_key_box = gr.Textbox(
        label="Anthropic API Key",
        placeholder="sk-ant-...",
        type="password",
        value=os.environ.get("ANTHROPIC_API_KEY", ""),
    )

    with gr.Row():
        data_type = gr.Dropdown(
            choices=list(PRESET_MAP.keys()),
            value="Student Records",
            label="Data Type",
        )
        num_records = gr.Slider(
            minimum=1, maximum=50, value=5, step=1,
            label="Number of Records",
        )

    custom_prompt = gr.Textbox(
        label="Custom Prompt (only used when Data Type = Custom)",
        placeholder="Generate 5 synthetic student ... records in JSON format. Include: ...",
        lines=5,
    )

    generate_btn = gr.Button("Generate Data", variant="primary")

    output_box = gr.Code(label="Generated JSON", language="json", lines=30)

    generate_btn.click(
        generate_ui,
        inputs=[api_key_box, data_type, custom_prompt, num_records],
        outputs=[output_box],
    )

print("Gradio UI built")

Gradio UI built


In [11]:
demo.launch(share=False)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
